In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# GBB and source-field clustering robustness

Calculate the SI S3.5.4–5 comparisons. The GBB experiment varies GBB construction with the source fields fixed. The source-field experiment varies source clustering with the GBBs fixed.

Read the reference profiles, CPC2vec model, layout vocabulary and `clusangle_ccmtagg.parquet` from `BASELINE`, and the rank-20 FSE projection from notebook 04. Set paths in the first cell. Save the two experiments under `OUTPUTS/gbb_parameters/` and `OUTPUTS/source_parameters/`.


In [ ]:
import pandas as pd
from gbb_utils import (read_profile, nonnoise, extra_codes, code_profile,
    reduce_profile, cluster_profile, correspondence, correspondence_metrics,
    native_network, gbb_crosswalk, align_gbbs, network_metrics, rca)
BASE = dict(neighbors=15, dimensions=5, min_size=100, iqr=1.5, n_matches=10)
CHANGES = {"neighbors": [5, 10, 30, 50], "dimensions": [2, 10],
           "min_size": [50, 75, 150, 200], "iqr": [1., 1.25, 1.75, 2.],
           "n_matches": [5, 8, 15, 20]}
OLD_NAMES = {"neighbors": "umap_neighbors", "dimensions": "umap_components",
             "min_size": "hdbscan_min_cluster_size", "iqr": "iqr_multiplier",
             "n_matches": "n_matches"}
from gbb_utils import ordinary_codes, source_crosswalk, align_sources


## GBB experiment: eighteen parameter changes


In [ ]:
def design():
    rows = []
    for key, values in CHANGES.items():
        for value in values:
            settings = dict(BASE, **{key: value})
            name = OLD_NAMES[key] + "_" + str(value).replace(".", "p")
            rows.append((name, settings))
    return rows

def submitted_network():
    edges = pd.read_parquet(BASELINE / "srcpath33.parquet")
    edges = edges.rename(columns={"src": "source", "path": "gbb", "ccmt_patent_id": "edge_count"})
    return rca(edges, "source", "gbb", "edge_count")


## Read the GBB experiment inputs

In [ ]:
from gensim.models import Word2Vec

out = OUTPUTS / "gbb_parameters"
out.mkdir(parents=True, exist_ok=True)
projection = pd.read_parquet(DATA / "embeddings/fse/green_proj_cosangle_top20.parquet")
reference = read_profile(BASELINE / "clusangle_outlier_hdbscan2.parquet")
sources = read_profile(BASELINE / "clusangle_outlier_hdbscan2_src2.parquet")
layout_codes = pd.read_parquet(BASELINE / "external/umap2d.parquet").cpc
vectors = Word2Vec.load(str(BASELINE / "external/cpc_sg_ns_dim50.model")).wv
ref_network = submitted_network()
blocks = sorted(nonnoise(reference).clus.unique())
profiles, layouts, rows = {}, {}, []


## Recompute GBBs and compare their networks

Reuse extracted profiles and UMAP coordinates across settings that share those calculations. HDBSCAN uses its default `min_samples` in this experiment. Exclude codes assigned to an alternative GBB from source roles in its network.


In [ ]:
for name, settings in design():
    profile_key = (settings["n_matches"], settings["iqr"])
    if profile_key not in profiles:
        profile = code_profile(extra_codes(projection, *profile_key))
        profiles[profile_key] = profile.loc[profile.cpc.isin(layout_codes)].reset_index(drop=True)
    profile = profiles[profile_key]
    layout_key = (*profile_key, settings["neighbors"], settings["dimensions"])
    if layout_key not in layouts:
        layouts[layout_key] = reduce_profile(profile, vectors, settings["neighbors"], settings["dimensions"], SEED)
    partition = cluster_profile(profile, layouts[layout_key], min_size=settings["min_size"])
    partition.to_parquet(out / f"{name}_partition.parquet", index=False)
    _, cells = correspondence(reference, partition)
    cells.to_csv(out / f"{name}_correspondence.csv", index=False)
    focal = projection.loc[projection["rank"] <= settings["n_matches"], ["ccmt_patent_id", "cpc"]].drop_duplicates()
    native = native_network(focal, partition, sources)
    mapping = gbb_crosswalk(reference, partition)
    native.to_parquet(out / f"{name}_native.parquet", index=False)
    mapping.to_parquet(out / f"{name}_crosswalk.parquet", index=False)
    aligned = align_gbbs(native, mapping)
    rows.append(dict(variant=name, **settings, clusters=nonnoise(partition).clus.nunique(),
                     **correspondence_metrics(cells), **network_metrics(ref_network, aligned, blocks)))
    print(name, rows[-1])
pd.DataFrame(rows).to_csv(out / "stability.csv", index=False)


## Source-field experiment: recluster the source strengths

Use the source profile's full code set, including noise rows, and its recorded means/counts. Fix `min_samples=100`, vary the settings below, and measure GBB degree on each resulting network.


In [ ]:
SOURCE_DESIGN = [("finer", 15, 50), ("coarser", 15, 200), ("neighbours30", 30, 100)]

from gensim.models import Word2Vec

out = OUTPUTS / "source_parameters"
out.mkdir(parents=True, exist_ok=True)
sources = read_profile(BASELINE / "clusangle_outlier_hdbscan2_src2.parquet").sort_values("cpc").reset_index(drop=True)
gbbs = read_profile(BASELINE / "clusangle_outlier_hdbscan2.parquet")
focal = pd.read_parquet(BASELINE / "clusangle_ccmtagg.parquet", columns=["ccmt_patent_id", "cpc"])
focal["cpc"] = ordinary_codes(focal.cpc)
focal = focal.drop_duplicates()
vectors = Word2Vec.load(str(BASELINE / "external/cpc_sg_ns_dim50.model")).wv
reference = submitted_network()
layouts, rows = {}, []
blocks = sorted(nonnoise(gbbs).clus.unique())


## Recluster sources, align their network counts, and compare

In [ ]:
for name, neighbors, min_size in SOURCE_DESIGN:
    if neighbors not in layouts:
        layouts[neighbors] = reduce_profile(sources, vectors, neighbors, 5, SEED)
    partition = cluster_profile(sources, layouts[neighbors], min_size, min_samples=100, prefix="s")
    native = native_network(focal, gbbs, partition)
    mapping = source_crosswalk(sources, partition, gbbs)
    aligned = align_sources(native, mapping)
    _, cells = correspondence(sources, partition)
    row = dict(variant=name, source_clusters=nonnoise(partition).clus.nunique(),
               noise_share=partition.clus.eq("s-1").mean(),
               nmi=correspondence_metrics(cells)["nmi"],
               **network_metrics(reference, aligned, blocks, degree_network=native))
    rows.append(row)
    partition.to_parquet(out / f"{name}_partition.parquet", index=False)
    native.to_parquet(out / f"{name}_native.parquet", index=False)
    mapping.to_parquet(out / f"{name}_crosswalk.parquet", index=False)
    cells.to_csv(out / f"{name}_correspondence.csv", index=False)
    print(row)
pd.DataFrame(rows).to_csv(out / "stability.csv", index=False)
